# Estoque e Vendas / Stock and Sales
**EN:** Stage 1 - synthetic data generator (100% fake data, fixed seed).
**PT:** Etapa 1 - gerador de dados sintéticos (100% fictícios, seed fixa).

In [ ]:
# EN: Install dependencies in the first cell so the notebook runs after a session restart.
# PT: Instala as dependências na primeira célula para o notebook rodar após reiniciar a sessão.
%pip install -q "numpy>=1.24" "pandas>=2.0" "matplotlib>=3.7"

In [ ]:
import numpy as np
import pandas as pd

# EN: Fixed seed = same data on every run (reproducible).
# PT: Seed fixa = mesmos dados em toda execução (reproduzível).
SEED: int = 42
N_CLIENTES: int = 200
N_VENDAS: int = 3000
ANO: int = 2025

# EN: Fake catalog: (name, category, sale price in BRL).
# PT: Catálogo fictício: (nome, categoria, preço de venda em R$).
CATALOGO: list[tuple[str, str, float]] = [
    ("Água mineral 500ml", "Bebidas", 2.50),
    ("Refrigerante 2L", "Bebidas", 9.00),
    ("Suco de laranja 1L", "Bebidas", 7.50),
    ("Café pronto 250ml", "Bebidas", 5.00),
    ("Chá gelado 1,5L", "Bebidas", 8.00),
    ("Energético 269ml", "Bebidas", 8.50),
    ("Pão francês 1kg", "Padaria", 16.00),
    ("Pão de forma", "Padaria", 8.50),
    ("Bolo de cenoura", "Padaria", 14.00),
    ("Biscoito recheado", "Padaria", 4.50),
    ("Croissant", "Padaria", 6.50),
    ("Rosca doce", "Padaria", 9.00),
    ("Leite 1L", "Laticínios", 5.50),
    ("Iogurte natural 170g", "Laticínios", 3.50),
    ("Queijo mussarela 200g", "Laticínios", 14.00),
    ("Manteiga 200g", "Laticínios", 12.00),
    ("Requeijão 200g", "Laticínios", 8.00),
    ("Creme de leite 200g", "Laticínios", 4.50),
    ("Arroz 5kg", "Mercearia", 28.00),
    ("Feijão 1kg", "Mercearia", 8.50),
    ("Macarrão 500g", "Mercearia", 5.00),
    ("Óleo de soja 900ml", "Mercearia", 7.50),
    ("Açúcar 1kg", "Mercearia", 5.50),
    ("Farinha de trigo 1kg", "Mercearia", 6.00),
    ("Detergente 500ml", "Limpeza", 2.80),
    ("Sabão em pó 1kg", "Limpeza", 12.00),
    ("Desinfetante 1L", "Limpeza", 6.50),
    ("Água sanitária 1L", "Limpeza", 5.00),
    ("Esponja multiuso", "Limpeza", 3.00),
    ("Papel higiênico 12un", "Limpeza", 19.00),
]

# EN: Cities with weights (bigger weight = more customers). Real cities, fake customers.
# PT: Cidades com pesos (peso maior = mais clientes). Cidades reais, clientes fictícios.
CIDADES: dict[str, float] = {
    "São Paulo": 0.40,
    "Guarulhos": 0.12,
    "Osasco": 0.10,
    "Santo André": 0.10,
    "São Bernardo do Campo": 0.08,
    "Campinas": 0.08,
    "Santos": 0.07,
    "Sorocaba": 0.05,
}

# EN: Month weights = seasonality (December sells more, February sells less).
# PT: Pesos por mês = sazonalidade (dezembro vende mais, fevereiro vende menos).
PESO_MES: dict[int, float] = {
    1: 1.0, 2: 0.8, 3: 0.9, 4: 0.95, 5: 1.0, 6: 0.9,
    7: 1.0, 8: 1.0, 9: 1.0, 10: 1.1, 11: 1.2, 12: 1.5,
}

In [ ]:
def gerar_produtos(rng: np.random.Generator) -> pd.DataFrame:
    """Build the products table / Monta a tabela de produtos."""
    # EN: Cost is 55%-80% of the price, so profit is always positive.
    # PT: Custo é 55%-80% do preço, então o lucro é sempre positivo.
    linhas = []
    for id_produto, (nome, categoria, preco) in enumerate(CATALOGO, start=1):
        custo = round(preco * rng.uniform(0.55, 0.80), 2)
        linhas.append((id_produto, nome, categoria, preco, custo))
    return pd.DataFrame(
        linhas, columns=["id", "nome", "categoria", "preco", "custo"]
    )


def gerar_clientes(rng: np.random.Generator, n: int) -> pd.DataFrame:
    """Build the customers table (no names) / Monta a tabela de clientes (sem nomes)."""
    # EN: Only id + city: no personal data at all (LGPD-safe).
    # PT: Só id + cidade: nenhum dado pessoal (seguro para a LGPD).
    cidades = rng.choice(
        list(CIDADES.keys()), size=n, p=list(CIDADES.values())
    )
    return pd.DataFrame({"id": np.arange(1, n + 1), "cidade": cidades})


def gerar_vendas(
    rng: np.random.Generator,
    produtos: pd.DataFrame,
    clientes: pd.DataFrame,
    n: int,
    ano: int,
) -> pd.DataFrame:
    """Build the sales table / Monta a tabela de vendas."""
    # EN: Each day gets a weight from its month, so sales have seasonality.
    # PT: Cada dia recebe o peso do seu mês, então as vendas têm sazonalidade.
    dias = pd.date_range(f"{ano}-01-01", f"{ano}-12-31", freq="D")
    pesos_dia = np.array([PESO_MES[d.month] for d in dias])
    pesos_dia = pesos_dia / pesos_dia.sum()
    datas = dias[rng.choice(len(dias), size=n, p=pesos_dia)]

    # EN: Uneven popularity: some products sell much more than others.
    # PT: Popularidade desigual: alguns produtos vendem bem mais que outros.
    popularidade = rng.lognormal(mean=0.0, sigma=0.4, size=len(produtos))
    popularidade = popularidade / popularidade.sum()

    vendas = pd.DataFrame(
        {
            # EN: ISO text (YYYY-MM-DD) because SQLite has no date type.
            # PT: Texto ISO (AAAA-MM-DD) porque o SQLite não tem tipo data.
            "data": datas.strftime("%Y-%m-%d"),
            "id_produto": rng.choice(produtos["id"], size=n, p=popularidade),
            "id_cliente": rng.choice(clientes["id"], size=n),
            "quantidade": rng.integers(1, 6, size=n),
        }
    )
    # EN: Sort by date first, then create ids, so ids follow time order.
    # PT: Ordena por data antes de criar os ids, para o id seguir a ordem do tempo.
    vendas = vendas.sort_values("data").reset_index(drop=True)
    vendas.insert(0, "id", np.arange(1, n + 1))
    return vendas


def gerar_estoque(rng: np.random.Generator, produtos: pd.DataFrame) -> pd.DataFrame:
    """Build the stock table / Monta a tabela de estoque."""
    n = len(produtos)
    return pd.DataFrame(
        {
            "id_produto": produtos["id"],
            "quantidade_atual": rng.integers(0, 81, size=n),
            "estoque_minimo": rng.integers(10, 41, size=n),
        }
    )

In [ ]:
def validar_dados(
    produtos: pd.DataFrame,
    clientes: pd.DataFrame,
    vendas: pd.DataFrame,
    estoque: pd.DataFrame,
) -> None:
    """Check basic data rules / Confere regras básicas dos dados."""
    # EN: Fail early if something is wrong, before building the database.
    # PT: Falha cedo se algo estiver errado, antes de montar o banco.
    assert (produtos["custo"] < produtos["preco"]).all(), "custo >= preco"
    assert vendas["id_produto"].isin(produtos["id"]).all(), "produto inexistente"
    assert vendas["id_cliente"].isin(clientes["id"]).all(), "cliente inexistente"
    assert estoque["id_produto"].isin(produtos["id"]).all(), "estoque órfão"
    assert not any(
        df.isnull().any().any() for df in (produtos, clientes, vendas, estoque)
    ), "há valores nulos"
    abaixo = (estoque["quantidade_atual"] < estoque["estoque_minimo"]).sum()
    assert abaixo > 0, "nenhum produto abaixo do mínimo (pergunta 3 ficaria vazia)"


# EN: One rng, always called in the same order = same result every run.
# PT: Um único rng, sempre chamado na mesma ordem = mesmo resultado em toda execução.
rng = np.random.default_rng(SEED)
produtos = gerar_produtos(rng)
clientes = gerar_clientes(rng, N_CLIENTES)
vendas = gerar_vendas(rng, produtos, clientes, N_VENDAS, ANO)
estoque = gerar_estoque(rng, produtos)

validar_dados(produtos, clientes, vendas, estoque)

# EN: Only synthetic rows are shown here, safe to keep in the notebook output.
# PT: Aqui só aparecem linhas sintéticas, seguro manter no output do notebook.
for nome_tabela, tabela in {
    "produtos": produtos, "clientes": clientes,
    "vendas": vendas, "estoque": estoque,
}.items():
    print(f"{nome_tabela}: {len(tabela)} linhas")
vendas.head()